In [25]:
# Basic import necessary for configuration.
# Uncomment logging to hide log information.

import os
from collections import defaultdict
import boto3
from botocore.config import Config
import fsspec
import warnings
import glob
#import logging

# Control logging level: INFO, WARNING, ERROR
# Run command logging.disable if want to hide logging
# ERROR messages.
#logging.disable(logging.ERROR)
warnings.simplefilter("ignore", RuntimeWarning)

# ---------------------------Set Processing Steps----------------------------
# Individual pipeline stages can be turned on/off here.  Note that a later
# stage won't be able to run unless data products have already been
# produced from the prior stage.

# Science processing.
dodet1 = True  # calwebb_detector1
dospec2 = True  # calwebb_spec2
dospec3 = True  # calwebb_spec3
doviz = True  # Visualize calwebb outputs

# Background Processing.
dodet1bg = False  # calwebb_detector1
dospec2bg = False  # calwebb_spec2 (needed for Master Background subtraction)

# How should background subtraction be done?
# Set one or None of the flags below.
# If none are set, data will not be background subtracted.
# If background subtraction is done in Spec2 it will be skipped in Spec3.
# Note: Master-background subtraction is for dedicated background observations.
# Dedicated backgrounds must be processed through spec2 first.
skip_master_bg = True  # True = do not run master-background subtraction in spec3.
skip_pixel_bg = False  # False = Run pixel-based background subtraction in spec2.

# ------------------------Set CRDS context and paths------------------------
# Each version of the calibration pipeline is associated with a specific CRDS
# context file. The pipeline will select the appropriate context file behind
# the scenes while running. However, if you wish to override the default context
# file and run the pipeline with a different context, you can set that using
# the CRDS_CONTEXT environment variable. Here, we show how this is done,
# although we leave the line commented out in order to use the default context.
# If you wish to specify a different context, uncomment the line below.
#os.environ['CRDS_CONTEXT'] = 'jwst_1322.pmap'  # CRDS context for 2.0.0

# Set CRDS cache directory to user home if not already set.
if os.getenv('CRDS_PATH') is None:
    os.environ['CRDS_PATH'] = os.path.join(os.path.expanduser('~'), '.crds_cache')

# Check whether the CRDS server URL has been set. If not, set it.
if os.getenv('CRDS_SERVER_URL') is None:
    os.environ['CRDS_SERVER_URL'] = 'https://jwst-crds.stsci.edu'

# Output the current CRDS path and server URL in use.
print('CRDS local filepath:', os.environ['CRDS_PATH'])
print('CRDS file server:', os.environ['CRDS_SERVER_URL'])

# Use the entire available screen width for this notebook.
from IPython.display import display, HTML, JSON
display(HTML("<style>.container { width:95% !important; }</style>"))

# ----------------------General Imports----------------------
import time
import glob
import json
import itertools
import numpy as np
from collections import defaultdict

# ----------------------Astropy Imports----------------------
# Astropy utilities for opening FITS files, downloading demo files, etc.
from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.visualization import ImageNormalize, ManualInterval, LogStretch
from astropy.visualization import LinearStretch, AsinhStretch, simple_norm

# ----------------------Astroquery Import----------------------
from astroquery.mast import Observations

# ----------------------Plotting Imports---------------------
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from jdaviz import Specviz2d

# ----------------------JWST Calibration Pipeline Imports----------------------
import jwst  # Import the base JWST and CRDS packages.
import crds
from crds.client import api
from stpipe import crds_client

# JWST pipelines (each encompassing many steps).
from jwst.pipeline import Detector1Pipeline  # calwebb_detector1
from jwst.pipeline import Spec2Pipeline  # calwebb_spec2
from jwst.pipeline import Spec3Pipeline  # calwebb_spec3
from jwst.extract_1d import Extract1dStep  # Extract1D Step

# JWST pipeline utilities.
from jwst import datamodels  # JWST datamodels.
from jwst.associations import asn_from_list as afl  # Tools for creating ASN files.
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase  # Lvl2 ASN file.
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base  # Lvl3 ASN file.

default_context = crds.get_default_context('jwst', state='build')
print("JWST Calibration Pipeline Version = {}".format(jwst.__version__))
print(f"Default CRDS Context for JWST Version {jwst.__version__}: {default_context}")
print(f"Using CRDS Context: {os.environ.get('CRDS_CONTEXT', default_context)}")

def filter_list(files, key, value):
    """
    Filter a list of files based on a specific FITS header key-value condition.

    Parameters
    ----------
    files : list
        List of file paths to FITS files.
    key : str
        The FITS header key to check.
    value : Any
        The value to match for the specified header key.

    Returns
    -------
    list : A list of file paths that satisfy the key-value condition.
    """
    return [file for file in files if fits.getheader(file).get(key) == value]
    
def get_matching(files, detector, filt, grating, fxd_slit, exp_type):
    """
    Filters a list of FITS files to find those with matching 
    detector, filter, and grating for a specified exposure type.

    Parameters
    ----------
    files : list of str
        Paths to FITS files to check.
    detector : str
        Expected value of the DETECTOR keyword.
    filt : str
        Expected value of the FILTER keyword.
    grating : str
        Expected value of the GRATING keyword.
    fxd_slit : str
        Fixed slit name.
    exp_type : str, optional
        The exposure type to match.

    Returns
    -------
    files_regular : list of str
        Files with matching configuration and IS_IMPRT == False or missing.
    files_imprint : list of str)
        Files with matching configuration and IS_IMPRT == True.
    """
    files_regular, files_imprint = [], []
    for file in files:
        # Skip if EXP_TYPE doesn't match the provided one.
        if fits.getval(file, 'EXP_TYPE') != exp_type:
            files_regular.append(file)
            continue
        # Check if DETECTOR, FILTER, and GRATING match
        detector_match = fits.getval(file, 'DETECTOR') == detector
        filter_match = fits.getval(file, 'FILTER') == filt
        grating_match = fits.getval(file, 'GRATING') == grating
        slit_match = fits.getval(file, 'FXD_SLIT') == fxd_slit
        if detector_match and filter_match and grating_match and slit_match:
            # Only IFU and MOS observations have imprint exposures.
            try:
                is_imprt = fits.getval(file, 'IS_IMPRT')
            except KeyError:
                is_imprt = None
            (files_imprint if is_imprt else files_regular).append(file)
    return files_regular, files_imprint
    
def match_gwa(file1, file2):
    """
    Check if GWA tilt values match closely enough to be associated.
    
    Parameters
    ----------
    file1, file2 : str 
        Input exposures FITS file paths.

    Returns
    -------
    True if both GWA tilt values match within tolerance, else False.
    """
    hdr1, hdr2 = fits.getheader(file1), fits.getheader(file2)
    return np.allclose(
        (hdr1['GWA_XTIL'], hdr1['GWA_YTIL']),
        (hdr2['GWA_XTIL'], hdr2['GWA_YTIL']),
        atol=1e-8, rtol=0
    )

CRDS local filepath: /home/dylan/.crds_cache
CRDS file server: https://jwst-crds.stsci.edu


JWST Calibration Pipeline Version = 2.0.0
Default CRDS Context for JWST Version 2.0.0: jwst_1535.pmap
Using CRDS Context: jwst_1535.pmap


## Enable cloud access

In [2]:
Observations.enable_cloud_dataset(provider='AWS')

# Uncomment line to disbale cloud access
# Observations.disable_cloud_dataset()

INFO: Using the S3 STScI public dataset [astroquery.mast.cloud]


## Define program details

In [3]:
dataproduct_type = "spectrum"
proposal_id = "4583"
proposal_pi = "Scholz, Aleks"

obs_table = Observations.query_criteria(
    dataproduct_type=dataproduct_type,
    proposal_id=proposal_id,
    instrument_name="NIRSPEC/SLIT",
    proposal_pi=proposal_pi,
)

unique_products = Observations.get_unique_product_list(obs_table)

filtered = Observations.filter_products(
    unique_products,
    calib_level="1",
    productType="SCIENCE",
    filters="CLEAR;PRISM",
)

INFO: 1583 of 2272 products were duplicates. Only returning 689 unique product(s). [astroquery.mast.utils]
INFO: To return all products, use `Observations.get_product_list` [astroquery.mast.observations]


In [4]:
# creating some mappings to keep directories organized
targets = obs_table['target_name']
obsids = obs_table['obsid']
obs_ids = filtered['obs_id'] # this is confusing
parent_obsid = filtered['parent_obsid']

obsid_to_target = {}
for i in range(len(obsids)):
    obsid_to_target[obsids[i]] = targets[i]
obsid_to_target = {str(k): str(v) for k, v in obsid_to_target.items()}

obs_id_to_parent_obsid = {}
for i in range(len(obs_ids)):
    obs_id_to_parent_obsid[obs_ids[i]] = parent_obsid[i]
obs_id_to_parent_obsid = {str(k): str(v) for k, v in obs_id_to_parent_obsid.items()}

obs_id_to_target = {k: obsid_to_target[v] for k, v in obs_id_to_parent_obsid.items()}

In [5]:
# Group uncal files by object
s3_uris = Observations.get_cloud_uris(filtered)
grouped_uris = defaultdict(list)

for uri in s3_uris:
    group_key = uri.split('/')[-1].strip('_uncal.fits')
    grouped_uris[obs_id_to_target[group_key]].append(uri)
grouped_uris

defaultdict(list,
            {'CHA1107-7626': ['s3://stpubdata/jwst/public/jw04583/jw04583018001/jw04583018001_03102_00001_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583018001/jw04583018001_03102_00003_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583018001/jw04583018001_03102_00002_nrs1_uncal.fits'],
             'CHA1110-7633': ['s3://stpubdata/jwst/public/jw04583/jw04583017001/jw04583017001_03102_00002_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583017001/jw04583017001_03102_00003_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583017001/jw04583017001_03102_00001_nrs1_uncal.fits'],
             'CHA1110-7721': ['s3://stpubdata/jwst/public/jw04583/jw04583016001/jw04583016001_03102_00003_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583016001/jw04583016001_03102_00002_nrs1_uncal.fits',
              's3://stpubdata/jwst/public/jw04583/jw04583016001/jw04

In [6]:
# Define output subdirectories to keep data products organized
base_dir = '/home/dylan/JWST/GO-4583/'
instrument = 'NIRSPEC'

In [7]:
bg_dir = ''  # If no background observation, use an empty string.

for obs in grouped_uris.keys():
    # Define/create output subdirectories to keep data products organized.
    
    # -----------------------------Science Directories------------------------------
    uncal_dir = os.path.join(base_dir, obs, instrument, '.uncal/')  # Uncalibrated pipeline inputs.
    asn_dir = os.path.join(base_dir, obs, instrument, 'asn/')  # Association files.
    det1_dir = os.path.join(base_dir, obs, instrument, 'stage1/')  # calwebb_detector1 pipeline outputs.
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2/')  # calwebb_spec2 pipeline outputs.
    spec3_dir = os.path.join(base_dir, obs, instrument, 'stage3/')  # calwebb_spec3 pipeline outputs.
    
    # Creates the directories if target directory does not exist.
    os.makedirs(uncal_dir, exist_ok=True)
    os.makedirs(det1_dir, exist_ok=True)
    os.makedirs(asn_dir, exist_ok=True)
    os.makedirs(spec2_dir, exist_ok=True)
    os.makedirs(spec3_dir, exist_ok=True)
    
    # ---------------------------Background Directories-----------------------------
    # if bg_dir:
    #     uncal_bgdir = os.path.join(base_dir, obs, instrument, 'uncal/')  # Uncalibrated pipeline inputs.
    #     asn_bgdir = os.path.join(base_dir, obs, instrument, 'asn/')  # Association files.
    #     det1_bgdir = os.path.join(base_dir, obs, instrument, 'stage1/')  # calwebb_detector1 pipeline outputs.
    #     spec2_bgdir = os.path.join(base_dir, obs, instrument, 'stage2/')  # calwebb_spec2 pipeline outputs.
    # 
    #     # Creates directories if background observations are provided and do not already exist.
    #     os.makedirs(det1_bgdir, exist_ok=True)
    #     os.makedirs(asn_bgdir, exist_ok=True)
    #     os.makedirs(spec2_bgdir, exist_ok=True)

## `Detector1Pipeline`

In [8]:
# Set up a dictionary to define how the Detector1 pipeline should be configured

# this sets up any entry to det1dict to be a dictionary itself
det1dict = defaultdict(dict)

# ---------------------------Override reference files---------------------------
# Example overrides for various reference files
#   Files should be in the base local directory or provide full path
#det1dict['dq_init']['override_mask'] = 'myfile.fits' # Bad pixel mask
#det1dict['saturation']['override_saturation'] = 'myfile.fits' # Saturation
#det1dict['superbias']['override_superbias'] = 'myfile.fits'  # Bias subtraction
#det1dict['refpix']['override_refpix'] = 'myfile.fits' # Reference pixel
#det1dict['linearity']['override_linearity'] = 'myfile.fits' # Linearity
#det1dict['dark_current']['override_dark'] = 'myfile.fits' # Dark current subtraction
#det1dict['jump']['override_gain'] = 'myfile.fits' # Gain used by jump step
#det1dict['ramp_fit']['override_gain'] = 'myfile.fits' # Gain used by ramp fitting step
#det1dict['jump']['override_readnoise'] = 'myfile.fits' # Read noise used by jump step
#det1dict['ramp_fit']['override_readnoise'] = 'myfile.fits' # Read noise used by ramp fitting step

# -----------------------------Set step parameters------------------------------
# Example overrides for whether or not certain steps should be skipped;
#det1dict['linearity']['skip'] = True  # skipping the linearity correction (default)

# Suppress computations for saturated ramps with only one good (unsaturated) sample (default True).
# The demo data has some saturation.
det1dict['ramp_fit']['suppress_one_group'] = True

# Turn on multi-core processing (off by default).
# Choose what fraction of cores to use (quarter, half, or all).
det1dict['jump']['maximum_cores'] = 'half'

The `refpix_algorithm` parameter in the `refpix` step is applicable only to NIR data. For NIRSpec full-frame data using traditional readout modes (NRS, NRSRAPID), the default algorithm—set in the parameter reference file as of Build 12.0—is 'sirs' (Simple Improved Reference Subtraction), which uses reference pixels from the left and right sides. For subarray data or non-traditional readouts, the algorithm automatically defaults to 'median'. Processing NIRSpec data, so leave as default setting.

In [9]:
#det1dict['refpix']['refpix_algorithm'] = 'sirs'  # default

Many exposures are affected by artifacts known as [snowballs](https://jwst-docs.stsci.edu/known-issues/shower-and-snowball-artifacts) caused by large cosmic ray events. These artifacts are particularly significant in deep exposures with long integration times, with an estimated rate of one snowball per detector (FULL FRAME) per 20 seconds. To expand the number of pixels flagged as jumps around large cosmic ray events, set `expand_large_events` to True. An `expand_factor` of 3 works well for NIRSpec observations to cover most snowballs.

In [10]:
# Turn on detection of cosmic ray snowballs (on by default).
det1dict['jump']['expand_large_events'] = True # default value
det1dict['jump']['expand_factor'] = 3 # default value

JWST detector readout electronics (a.k.a. SIDECAR ASICs) generate significant 1/f noise during detector operations and signal digitization. This noise manifests as faint banding along the detector's slow axis and varies from column to column. For NIRSpec data, the `clean_flicker_noise` step is the primary pipeline algorithm to address 1/f noise. This step can be run at the group level in the `Detector1Pipeline` or at the exposure level in the `Spec2Pipeline`. 
    
When the step is run within the `Spec2Pipeline` with the following parameters, it is closest to the `nsclean` algorithm (Rauscher 2023):
- fit_method: "fft"
- background_method: None
- mask_science_regions: True
- n_sigma: 5.0

Note that these values, especially the FFT fit method, can be extremely scene-dependent. The NIRSpec default values for clean_flicker_noise
parameters are instead:
- fit_method: "median"
- background_method: None
- mask_science_regions: False
- n_sigma: 1.5

The `clean_flicker_noise` step is currently off by default in both `Detector1Pipeline` and `Spec2Pipeline`, but can be turned on by setting "skip" to "False". Going to compare on vs off with this dataset, see if I get a better SNR. Using default values.

In [11]:
# Turn on 1/f noise correction in Stage 1 (off by default).
det1dict['clean_flicker_noise']['skip'] = False
# det1dict['clean_flicker_noise']['fit_method'] = 'fft'
# det1dict['clean_flicker_noise']['background_method'] = None
# det1dict['clean_flicker_noise']['mask_science_regions'] = True
# det1dict['clean_flicker_noise']['n_sigma'] = 5

In [12]:
# stage 1 for all targets
# Loop over each target
for obs, s3_uris in grouped_uris.items():

    uncal_dir = os.path.join(base_dir, obs, instrument, '.uncal')  # temporary uncal files will go here
    det1_dir = os.path.join(base_dir, obs, instrument, 'stage1')   # calwebb_detector1 pipeline outputs will go here

    print(f"\nStage 1 for {obs}...")
    
    # Download and process the files for this target object
    for uri in s3_uris:
        filename = os.path.basename(uri)
        uncal_path = os.path.join(uncal_dir, filename)

        stage1_wildcard = filename.replace("_uncal", "*")
        stage1_product_matches = glob.glob(os.path.join(det1_dir, stage1_wildcard))

        if stage1_product_matches:
            print(f"Found Stage 1 products for {filename}... Skipping...")
            continue 

        if os.path.exists(uncal_path):
            print(f"Found local uncal file {filename}...")
            
        # Following astroquery
        else:
            print(f"Downloading: {filename}")
            with fsspec.open(uri, "rb", anon=True, s3_additional_kwargs={'RequestPayer': 'requester'}) as cloud_file:
                with open(uncal_path, "wb") as local_file:
                    local_file.write(cloud_file.read())
        
        print(f"Running Detector1Pipeline on {filename}...")
        Detector1Pipeline.call(uncal_path, steps=det1dict, save_results=True, output_dir=det1_dir)

        # Clean raw file
        os.remove(uncal_path)

print("\nAll files successfully processed through Stage 1!")


Stage 1 for CHA1107-7626...
Downloading: jw04583018001_03102_00001_nrs1_uncal.fits
Running Detector1Pipeline on jw04583018001_03102_00001_nrs1_uncal.fits...


2026-05-17 09:53:12,160 - CRDS - INFO -  Calibration SW Found: jwst 2.0.0 (/home/dylan/.nirspec_fs_pipeline/lib/python3.12/site-packages/jwst-2.0.0.dist-info)
2026-05-17 09:53:13,216 - stpipe.step - INFO - PARS-DARKCURRENTSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-darkcurrentstep_0005.asdf
2026-05-17 09:53:13,233 - stpipe.step - INFO - PARS-JUMPSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-jumpstep_0004.asdf
2026-05-17 09:53:13,251 - stpipe.step - INFO - PARS-CLEANFLICKERNOISESTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf
2026-05-17 09:53:13,263 - stpipe.pipeline - INFO - PARS-DETECTOR1PIPELINE parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-detector1pipeline_0004.asdf
2026-05-17 09:53:13,286 - stpipe.step - INFO - Detector1Pipeline instance created.
2026-05-17 09:53:13,287 - stpipe.step - 

Downloading: jw04583018001_03102_00003_nrs1_uncal.fits


2026-05-17 09:54:39,102 - stpipe.step - INFO - PARS-DARKCURRENTSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-darkcurrentstep_0005.asdf
2026-05-17 09:54:39,116 - stpipe.step - INFO - PARS-JUMPSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-jumpstep_0004.asdf
2026-05-17 09:54:39,130 - stpipe.step - INFO - PARS-CLEANFLICKERNOISESTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf
2026-05-17 09:54:39,140 - stpipe.pipeline - INFO - PARS-DETECTOR1PIPELINE parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-detector1pipeline_0004.asdf
2026-05-17 09:54:39,162 - stpipe.step - INFO - Detector1Pipeline instance created.
2026-05-17 09:54:39,163 - stpipe.step - INFO - GroupScaleStep instance created.
2026-05-17 09:54:39,164 - stpipe.step - INFO - DQInitStep instance created.
2026-05-17 09:54:39,166 - stpipe.step - INF

Running Detector1Pipeline on jw04583018001_03102_00003_nrs1_uncal.fits...


...

Downloading: jw04583010001_03102_00003_nrs1_uncal.fits


2026-05-17 10:26:57,453 - stpipe.step - INFO - PARS-DARKCURRENTSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-darkcurrentstep_0005.asdf
2026-05-17 10:26:57,467 - stpipe.step - INFO - PARS-JUMPSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-jumpstep_0004.asdf
2026-05-17 10:26:57,484 - stpipe.step - INFO - PARS-CLEANFLICKERNOISESTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf
2026-05-17 10:26:57,496 - stpipe.pipeline - INFO - PARS-DETECTOR1PIPELINE parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-detector1pipeline_0004.asdf
2026-05-17 10:26:57,516 - stpipe.step - INFO - Detector1Pipeline instance created.
2026-05-17 10:26:57,517 - stpipe.step - INFO - GroupScaleStep instance created.
2026-05-17 10:26:57,518 - stpipe.step - INFO - DQInitStep instance created.
2026-05-17 10:26:57,519 - stpipe.step - INF

Running Detector1Pipeline on jw04583010001_03102_00003_nrs1_uncal.fits...


2026-05-17 10:26:57,704 - stpipe.step - INFO - Step Detector1Pipeline running with args ('/home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/.uncal/jw04583010001_03102_00003_nrs1_uncal.fits',).
2026-05-17 10:26:57,727 - stpipe.step - INFO - Step Detector1Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/stage1
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_calibrated_ramp: False
  steps:
    group_scale:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
    dq_init:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None


All files successfully processed through Stage 1!


## `Spec2Pipeline`

In [13]:
# Set up a dictionary to define how the Spec2 pipeline should be configured.

# this sets up any entry to spec2dict to be a dictionary itself
spec2dict = defaultdict(dict)

# ---------------------------Override reference files---------------------------
#spec2dict['assign_wcs']['override_wavelengthrange'] = 'myfile.asdf' # Wavelength channel mapping (ASDF file)
#spec2dict['extract_2d']['override_wavelengthrange'] = 'myfile.asdf' # Wavelength channel mapping (ASDF file)
#spec2dict['wavecorr']['override_wavecorr'] = 'myfile.fits' # Wavelength zero point correction
#spec2dict['flat_field']['override_fflat'] = 'myfile.fits' # Fore optics flat field
#spec2dict['flat_field']['override_sflat'] = 'myfile.fits' # Spectrograph flat field
#spec2dict['pathloss']['override_pathloss'] = 'myfile.fits' # Path loss correction
#spec2dict['extract_1d']['override_extract1d'] = 'myfile.json' # 1D spectral extraction parameters (JSON file)

# -----------------------------Set step parameters------------------------------
# Overrides for whether or not certain steps should be skipped (example).
spec2dict['bkg_subtract']['skip'] = skip_pixel_bg  # Runs if pixel-to-pixel bkg selected above.

# Run pixel replacement code to extrapolate values for otherwise bad pixels.
# This can help mitigate 5-10% negative dips in spectra of bright sources.
# Use the 'mingrad' algorithm.
#spec2dict['pixel_replace']['skip'] = False
#spec2dict['pixel_replace']['algorithm'] = 'mingrad'

# Turn on bad pixel self-calibration, where all exposures on a given detector 
# are used to find and flag bad pixels that may have been missed by the bad pixel mask.
# This step is experimental, and works best when dedicated background observations are included.
#spec2dict['badpix_selfcal']['skip'] = False
#spec2dict['badpix_selfcal']['flagfrac_upper'] = 0.005  # Fraction of pixels to flag.

Resampling 2D spectra can sometimes introduce artificial noise and reduce the signal-to-noise ratio (SNR) in the resulting 1D spectra when using `weight_type='ivm'` ([see known issues](https://jwst-docs.stsci.edu/known-issues-with-jwst-data/nirspec-known-issues/nirspec-fs-known-issues#NIRSpecFSKnownIssues-Resamplingof2-Dspectra:~:text=noise%20workaround%20notebook.-,Resampling%20of%202%2DD%20spectra,-Stages%202%20and)). The default is now set to set weight type to 'exptime'. Consider the following when selecting a `weight_type` parameter:
* **'ivm'**: Inverse variant scaling based on read noise (VAR_RNOISE), ideal for rejecting outliers and better suited for faint sources.
* **'exptime'**: Uses exposure time for scaling, improving SNR for bright sources.

I have tried using `ivm` instead of `exptime` and final results are useless, (there was no flux error, so something must have broke). I'll play around with it again here, see what happens.

In [14]:
# Resample weight_type.
spec2dict['resample_spec']['weight_type'] = 'ivm'

To correct for 1/f noise with `clean_flicker_noise` in Stage 2, see the **FS_NSClean_example** demo notebook for FS data [here](https://github.com/spacetelescope/jdat_notebooks/tree/main/notebooks/NIRSpec/NIRSpec_NSClean). The following parameters give results closest to the NSClean algorithm. This correction can be highly scene dependent.

For my test, I'll do this step for the `Detector1Pipeline` and not here. Maybe more testing can be done to see if turning it on for this step instead of `Detector1` gives different results.

In [16]:
# Run clean_flicker_noise for 1/f noise on the rate files.
spec2dict['clean_flicker_noise']['skip'] = True
#spec2dict['clean_flicker_noise']['fit_method'] = "fft"
#spec2dict['clean_flicker_noise']['background_method'] = None
#spec2dict['clean_flicker_noise']['mask_science_regions'] = True
#spec2dict['clean_flicker_noise']['n_sigma'] = 5

Below is functions for the association stuff. This is how the pipeline takes multiple exposures and combines into one product to acheive hight SNR. [Read more here.](https://jwst-pipeline.readthedocs.io/en/stable/jwst/associations/overview.html)

In [17]:
def asn_nod(asn, onescifile, sci, sci_imprint, nod_num):
    """
    Associate background, imprint, and selfcal exposures for nodded observations.

    Parameters
    ----------
    asn : dict
        The association dictionary to update.
    onescifile : str 
        Path to the primary science file.
    sci : list of str
        List of science exposure file paths.
    sci_imprint : list of str
        List of science imprint exposure file paths.
    nod_num : int
        Position in dither pattern.

    Returns
    -------
    asn : dict 
        Updated association dictionary with members for applicable background, imprint, and selfcal.
    """
    members = asn['products'][0]['members']

    # Assign background exposures.
    for file in sci:
        # If dither position is different from the input position, use it as a background.
        if ((fits.getval(file, 'PATT_NUM') - 1) // (fits.getval(file, 'SUBPXPTS'))) != nod_num:
            members.append({'expname': file, 'exptype': 'background'})
    
    # Assign imprint exposures (pipeline handles figuring out which one is best).
    for file in sci_imprint:
        # Only IFU and MOS observations have imprint exposures.
        if fits.getval(file, 'EXP_TYPE') == 'NRS_IFU' or 'NRS_MSASPEC':
            if match_gwa(onescifile, file):
                members.append({'expname': file, 'exptype': 'imprint'})

    # Assign selfcal exposures.
    for file in sci + sci_imprint:
        members.append({'expname': file, 'exptype': 'selfcal'})

    return asn

def asn_dither(asn, onescifile, sci, sci_imprint, bg, bg_imprint):
    """
    Associate background, imprint, and selfcal exposures for dithered observations.

    Parameters
    ----------
    asn : dict
        The association dictionary to update.
    onescifile : str 
        Path to the primary science file.
    sci : list of str
        List of science exposure file paths.
    sci_imprint : list of str
        List of science imprint exposure file paths.
    bg : list of str
        List of background exposure file paths.
    bg_imprint : list of str
        List of background imprint exposure file paths.

    Returns
    -------
    asn : dict 
        Updated association dictionary with members for applicable background, imprint, and selfcal.
    """
    members = asn['products'][0]['members']
    
    # Assign background exposures.
    for file in bg:
        members.append({'expname': file, 'exptype': 'background'})

    # Assign imprint exposures (pipeline handles figuring out which one is best).
    for file in sci_imprint:
        # Only IFU and MOS observations have imprint exposures.
        if fits.getval(file, 'EXP_TYPE') == 'NRS_IFU' or 'NRS_MSASPEC':
            if match_gwa(onescifile, file):
                members.append({'expname': file, 'exptype': 'imprint'})
    for file in bg_imprint:
        # Only IFU and MOS observations have imprint exposures.
        if fits.getval(file, 'EXP_TYPE') == 'NRS_IFU' or 'NRS_MSASPEC':
            if match_gwa(bg[0], file):
                members.append({'expname': file, 'exptype': 'imprint'})

    # Assign selfcal exposures.
    for file in sci + sci_imprint + bg + bg_imprint:
        members.append({'expname': file, 'exptype': 'selfcal'})
    
    return asn

def writel2asn(onescifile, allscifiles, bgfiles, asnfile, product_name, exp_type):
    """
    Create a Level 2 association file for each science exposure.

    Parameters
    ----------
    onescifile : str
        Path to the primary science exposure file.
    allscifiles : list of str
        List of all science exposure files.
    bgfiles : list of str
        List of background exposure files.
    asnfile : str
        Path to write the output association file.
    product_name : str
        Name of the product for the association.
    exp_type : str, optional
        Exposure type to match against.

    Returns
    -------
    True if the association was written successfully, and False otherwise 
    """
    # Define a basic association with the science file.
    # Wrap in array since input was single exposure.
    asn = afl.asn_from_list([onescifile], rule=DMSLevel2bBase, product_name=product_name)
    program = fits.getval(onescifile, 'PROGRAM')
    asn.data['program'] = program

    # Grab header information from the science file.
    exp_type = fits.getval(onescifile, 'EXP_TYPE')
    if (exp_type == exp_type):
        detector = fits.getval(onescifile, 'DETECTOR')
        grating = fits.getval(onescifile, 'GRATING')
        filt = fits.getval(onescifile, 'FILTER')
        fxd_slit = fits.getval(onescifile, 'FXD_SLIT')
        patttype = fits.getval(onescifile, 'PATTTYPE')  # Dither pattern type.
        pattnum = fits.getval(onescifile, 'PATT_NUM')  # Dither pattern number.
        # primary_dithpts = fits.getval(onescifile, 'PRIDTPTS')  # Points in primary dither pattern.
        subpxpts = fits.getval(onescifile, 'SUBPXPTS')  # Points in subpixel dither pattern.
        # Position number within primary dither pattern.
        nod_num = (pattnum - 1) // (subpxpts)

    # If the exposure type does not match, fail out 
    # to ensure TA images don't get processed by accident.
    else:
        return False

    # Find all files matching the input configuration and split into regular/imprint.
    use_sci, _ = get_matching(allscifiles, detector, filt, grating, fxd_slit, exp_type)
    use_bg, _ = get_matching(bgfiles, detector, filt, grating, fxd_slit, exp_type) if bgfiles else ([], [])

    # If this uses nodded exposures set up pixel-based background subtraction accordingly.
    is_nod = 'NOD' in patttype.split('-')
    if is_nod:
        asn = asn_nod(asn, onescifile, use_sci, [], nod_num)
    else:  # Otherwise handle as dithered exposures.
        asn = asn_dither(asn, onescifile, use_sci, [], use_bg, [])

    # Write the association to a json file.
    _, serialized = asn.dump()
    with open(asnfile, 'w') as outfile:
        outfile.write(serialized)
        
    return True

Need to modify this for my workflow

I think I just need to write a function that will go through each `stage1` directory, grab the `_rate.fits` files and then put them in a list. Will loop over each target. No background observations right now so just leaving that out for now.

In [ ]:
# To save on runtime turns off creation of quicklook 2d/1d spectra for science data.
# Any master background subtraction in spec3 will require the 1d spectra from spec2.
#spec2dict['resample_spec']['skip'] = True  # S2D products.
#spec2dict['extract_1d']['skip'] = True  # X1D products.

In [27]:
rate_bg = []

for obs in grouped_uris.keys():

    print(f"Stage 2 for {obs}...")
    det1_dir = os.path.join(base_dir, obs, instrument, 'stage1')
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2/')
    path = os.path.join(det1_dir, '*_rate.fits')

    rate_sci = glob.glob(path)

    # Special case for FS S1600A1 with a 5-POINT-NOD.
    # Exclude the top and bottom dithers [1, 5] that are close to slit edges.
    excluded_files = [
        file for file in rate_sci
        # Considering number of points in primary dither pattern and
        # Position number within primary dither pattern.
        if (
            (header := fits.getheader(file)).get('FXD_SLIT') == 'S1600A1' and header.get('PRIDTPTS') == 5 and (header.get('PATT_NUM') - 1) // header.get('SUBPXPTS') in [1, 5]
        )
    ]

    rate_sci = [file for file in rate_sci if file not in excluded_files]

    # Run Stage 2 pipeline using the custom spec2dict dictionary.
    if dospec2:
    
        # --------------------------Science files--------------------------
        for file in rate_sci:
            try:  # Create ASN files.
                asnfile = os.path.join(asn_dir, os.path.basename(file).replace('rate.fits', 'l2asn.json'))
                if writel2asn(file, rate_sci, rate_bg, asnfile, 'Level2', 'NRS_FIXEDSLIT'):
                    print(f"Applying Stage 2 Corrections & Calibrations to: {file}")
                    spec2sci_result = Spec2Pipeline.call(asnfile,
                                                         save_results=True,
                                                         steps=spec2dict,
                                                         output_dir=spec2_dir)
            except Exception as e:
                # A handle for when no slits fall on NRS2.
                print(f"Skipped processing {os.path.basename(asnfile)}: {e}")
        print(f"Spec2 has been completed for {obs}! \n")
    else:
        print(f'Skipping Spec2 processing for {obs}...')

/home/dylan/.nirspec_fs_pipeline/lib/python3.12/site-packages/jwst/associations/association.py:232: UserWarning: Input association file contains path information; note that this can complicate usage and/or sharing of such files.
  warnings.warn(err_str, UserWarning, stacklevel=1)
2026-05-17 11:46:13,087 - stpipe.step - INFO - PARS-CLEANFLICKERNOISESTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-cleanflickernoisestep_0001.asdf
2026-05-17 11:46:13,104 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-17 11:46:13,116 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-05-17 11:46:13,132 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-17 11:

Applying Stage 2 Corrections & Calibrations to: /home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage1/jw04583018001_03102_00003_nrs1_rate.fits


2026-05-17 11:46:13,564 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree: None
      sip_max_inv_pix_error: 0.01
      sip_inv_degree: None
      sip_npoints: 12
      slit_y_low: -0.55
      slit_y_high: 0.55
      nrs_ifu_slice_wcs: False
    badpix_selfc

Applying Stage 2 Corrections & Calibrations to: /home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage1/jw04583018001_03102_00001_nrs1_rate.fits


2026-05-17 11:46:34,713 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree: None
      sip_max_inv_pix_error: 0.01
      sip_inv_degree: None
      sip_npoints: 12
      slit_y_low: -0.55
      slit_y_high: 0.55
      nrs_ifu_slice_wcs: False
    badpix_selfc

...

Applying Stage 2 Corrections & Calibrations to: /home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/stage1/jw04583010001_03102_00001_nrs1_rate.fits


2026-05-17 22:18:29,460 - stpipe.step - INFO - Step Spec2Pipeline running with args ('/home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/asn/jw04583010001_03102_00001_nrs1_l2asn.json',).
2026-05-17 22:18:29,501 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/stage2/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree: N

Applying Stage 2 Corrections & Calibrations to: /home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/stage1/jw04583010001_03102_00003_nrs1_rate.fits


2026-05-17 22:18:50,164 - stpipe.step - INFO - Step Spec2Pipeline running with args ('/home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/asn/jw04583010001_03102_00003_nrs1_l2asn.json',).
2026-05-17 22:18:50,206 - stpipe.step - INFO - Step Spec2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: /home/dylan/JWST/GO-4583/UGC0433+2251/NIRSPEC/stage2/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  fail_on_exception: True
  save_wfss_esec: False
  steps:
    assign_wcs:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      sip_approx: True
      sip_max_pix_error: 0.01
      sip_degree: N

Spec2 has been completed for UGC0433+2251! 



In [31]:
for obs in grouped_uris.keys():
    
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2/')

    # List the Stage 2 products.
    print(f"Stage 2 products for {obs}:")
    
    # -----------------------------Science files-----------------------------
    sci_cal = sorted(glob.glob(spec2_dir + '*_cal.fits'))
    sci_s2d = sorted(glob.glob(spec2_dir + '*_s2d.fits'))
    sci_x1d = sorted(glob.glob(spec2_dir + '*_x1d.fits'))
    
    print(f"SCIENCE | Stage 2 CAL Products:\n{'-'*20}\n" + "\n".join(sci_cal))
    print(f"SCIENCE | Stage 2 S2D Products:\n{'-'*20}\n" + "\n".join(sci_s2d))
    print(f"SCIENCE | Stage 2 X1D Products:\n{'-'*20}\n" + "\n".join(sci_x1d) + "\n\n")
    
    if dospec2bg:
        # ----------------------------Background files---------------------------
        bg_cal = sorted(glob.glob(spec2_bgdir + '*_cal.fits'))
        bg_s2d = sorted(glob.glob(spec2_bgdir + '*_s2d.fits'))
        bg_x1d = sorted(glob.glob(spec2_bgdir + '*_x1d.fits'))
    
        print(f"BACKGROUND | Stage 2 CAL Products:\n{'-'*20}\n" + "\n".join(bg_cal))
        print(f"BACKGROUND | Stage 2 S2D Products:\n{'-'*20}\n" + "\n".join(bg_s2d))
        print(f"BACKGROUND | Stage 2 X1D Products:\n{'-'*20}\n" + "\n".join(bg_x1d))

Stage 2 products for CHA1107-7626:
SCIENCE | Stage 2 CAL Products:
--------------------
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00001_nrs1_cal.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00002_nrs1_cal.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00003_nrs1_cal.fits
SCIENCE | Stage 2 S2D Products:
--------------------
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00001_nrs1_s2d.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00002_nrs1_s2d.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00003_nrs1_s2d.fits
SCIENCE | Stage 2 X1D Products:
--------------------
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00001_nrs1_x1d.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw04583018001_03102_00002_nrs1_x1d.fits
/home/dylan/JWST/GO-4583/CHA1107-7626/NIRSPEC/stage2/jw0458301

## `Spec3Pipeline`

In [32]:
# Set up a dictionary to define how the Spec3 pipeline should be configured.

# this sets up any entry to spec3dict to be a dictionary itself
spec3dict = defaultdict(dict)

# ---------------------------Override reference files---------------------------
# Overrides for various reference files.
# Files should be in the base local directory or provide full path.
#spec3dict['extract_1d']['override_extract1d'] = 'myfile.json' (JSON file)

# -----------------------------Set step parameters------------------------------
# Overrides for whether or not certain steps should be skipped (example).
#spec3dict['outlier_detection']['skip'] = True

# Master background usage was set up above, propagate that here.
spec3dict['master_background']['skip'] = skip_master_bg

# Run pixel replacement code to extrapolate values for otherwise bad pixels.
# This can help mitigate 5-10% negative dips in spectra of bright sources.
# Use the 'mingrad' algorithm.
#spec3dict['pixel_replace']['skip'] = False
#spec3dict['pixel_replace']['algorithm'] = 'mingrad'

# Resample weight_type. Set to 'ivm' above, if that doesn't give good results I'll switch to doing that step here instead.
#spec3dict['resample_spec']['weight_type'] = 'exptime'

In [35]:
def writel3asn(scifiles, bgfiles, obs):
    """
    Create a Level 3 association file.

    Parameters
    ----------
    scifiles : list of str
        List of all science exposure files.
    bgfiles : list of str
        List of background exposure files.

    Returns
    -------
    None.
    """
    # Set asn directory
    asn_dir = os.path.join(base_dir, obs, instrument, 'asn/')
    # Filter based on GRATING/FILTER.
    grouped = defaultdict(lambda: {'sci': [], 'bg': []})

    for f in scifiles:
        k = (fits.getval(f, 'FILTER'), fits.getval(f, 'GRATING'), fits.getval(f, 'FXD_SLIT'))
        grouped[k]['sci'].append(f)
    for f in bgfiles:
        k = (fits.getval(f, 'FILTER'), fits.getval(f, 'GRATING'), fits.getval(f, 'FXD_SLIT'))
        grouped[k]['bg'].append(f)

    # Check for S200A1+S200A2 cases.
    to_delete = []
    for (filt, grat, slit), files in list(grouped.items()):
        if slit == 'S200A1':

            # Only continue if S200A2 group exists.
            key_a1 = (filt, grat, 'S200A1')
            key_a2 = (filt, grat, 'S200A2')
            if key_a2 not in grouped:
                continue

            # Collect all target names in each slit.
            targs_a1 = {fits.getval(fn, 'TARGNAME')
                        for fn in grouped[key_a1]['sci'] + grouped[key_a1]['bg']}
            targs_a2 = {fits.getval(fn, 'TARGNAME')
                        for fn in grouped[key_a2]['sci'] + grouped[key_a2]['bg']}
            
            # Only combine if both slits have a single, matching target.
            if len(targs_a1) == 1 and targs_a1 == targs_a2:
                new_key = (filt, grat, 'S200A1_S200A2')
                for kind in ('sci', 'bg'):
                    for fn in grouped[key_a1][kind] + grouped[key_a2][kind]:
                        grouped[new_key][kind].append(fn)
                to_delete += [key_a1, key_a2]

    # Remove old keys.
    for k in to_delete:
        grouped.pop(k, None)

    # Make ASN for each FILTER/GRATING.
    for (filt, grat, slit), files in grouped.items():
        name = f"{filt}_{grat}".lower()
        asnfile = os.path.join(asn_dir, f"{name}_l3asn.json")
        asn = afl.asn_from_list(files['sci'], rule=DMS_Level3_Base, product_name=name)
        for bg in files['bg']:
            asn['products'][0]['members'].append({'expname': bg, 'exptype': 'background'})
        with open(asnfile, 'w') as f:
            f.write(asn.dump()[1])
    print("Level 3 ASN creation complete!")

In [36]:
for obs in grouped_uris.keys():
    print(f"Writing .asn file for {obs}...")
    
    spec2_dir = os.path.join(base_dir, obs, instrument, 'stage2')
    path = os.path.join(spec2_dir, '*_cal.fits')

    scifiles = glob.glob(path)
    
    if dospec3:
        writel3asn(sci_cal, bg_x1d if bg_dir else [], obs)


Writing .asn file for CHA1107-7626...
Level 3 ASN creation complete!
Writing .asn file for CHA1110-7633...
Level 3 ASN creation complete!
Writing .asn file for CHA1110-7721...
Level 3 ASN creation complete!
Writing .asn file for UHWJ247.95-24.78...
Level 3 ASN creation complete!
Writing .asn file for UGC0422+2655...
Level 3 ASN creation complete!
Writing .asn file for UGC0417+2832...
Level 3 ASN creation complete!
Writing .asn file for UGC0439+2642...
Level 3 ASN creation complete!
Writing .asn file for UGC0433+2251...
Level 3 ASN creation complete!


In [37]:
for obs in grouped_uris.keys():

    print(f"Stage 3 for {obs}...")
    
    spec3_dir = os.path.join(base_dir, obs, instrument, 'stage3')
    asn_dir = os.path.join(base_dir, obs, instrument, 'asn/')

    spec3_asn = glob.glob(f"{asn_dir}*l3asn.json")
    # Run Stage 3 pipeline using the custom spec3dict dictionary.
    if dospec3:
    
        # --------------------------Spec3 ASN files--------------------------
        for s3_asn in spec3_asn:
            print(f"Applying Stage 3 Corrections & Calibrations to: "
                  f"{os.path.basename(s3_asn)}")
    
            spec3_result = Spec3Pipeline.call(s3_asn,
                                              save_results=True,
                                              steps=spec3dict,
                                              output_dir=spec3_dir)
        print("Spec3 has been completed! \n")
    else:
        print("Skipping Spec3. \n")

2026-05-18 07:25:08,400 - py.warnings - WARNING - /home/dylan/.nirspec_fs_pipeline/lib/python3.12/site-packages/jwst/associations/association.py:232: UserWarning: Input association file contains path information; note that this can complicate usage and/or sharing of such files.
  warnings.warn(err_str, UserWarning, stacklevel=1)



Stage 3 for CHA1107-7626...
Applying Stage 3 Corrections & Calibrations to: clear_prism_l3asn.json


2026-05-18 07:25:09,002 - stpipe.step - INFO - PARS-OUTLIERDETECTIONSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-outlierdetectionstep_0007.asdf
2026-05-18 07:25:09,061 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-18 07:25:09,145 - stpipe.step - INFO - Spec3Pipeline instance created.
2026-05-18 07:25:09,146 - stpipe.step - INFO - AssignMTWcsStep instance created.
2026-05-18 07:25:09,147 - stpipe.step - INFO - MasterBackgroundStep instance created.
2026-05-18 07:25:09,149 - stpipe.step - INFO - OutlierDetectionStep instance created.
2026-05-18 07:25:09,151 - stpipe.step - INFO - AdaptiveTraceModelStep instance created.
2026-05-18 07:25:09,152 - stpipe.step - INFO - PixelReplaceStep instance created.
2026-05-18 07:25:09,155 - stpipe.step - INFO - ResampleSpecStep instance created.
2026-05-18 07:25:09,158 - stpipe.step - INFO - C

Spec3 has been completed! 

Stage 3 for CHA1110-7633...
Applying Stage 3 Corrections & Calibrations to: clear_prism_l3asn.json


2026-05-18 07:25:40,079 - stpipe.step - INFO - PARS-OUTLIERDETECTIONSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-outlierdetectionstep_0007.asdf
2026-05-18 07:25:40,121 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-18 07:25:40,197 - stpipe.step - INFO - Spec3Pipeline instance created.
2026-05-18 07:25:40,198 - stpipe.step - INFO - AssignMTWcsStep instance created.
2026-05-18 07:25:40,199 - stpipe.step - INFO - MasterBackgroundStep instance created.
2026-05-18 07:25:40,200 - stpipe.step - INFO - OutlierDetectionStep instance created.
2026-05-18 07:25:40,201 - stpipe.step - INFO - AdaptiveTraceModelStep instance created.
2026-05-18 07:25:40,202 - stpipe.step - INFO - PixelReplaceStep instance created.
2026-05-18 07:25:40,204 - stpipe.step - INFO - ResampleSpecStep instance created.
2026-05-18 07:25:40,206 - stpipe.step - INFO - C

...

Spec3 has been completed! 

Stage 3 for UGC0439+2642...
Applying Stage 3 Corrections & Calibrations to: clear_prism_l3asn.json


2026-05-18 07:28:06,461 - stpipe.step - INFO - PARS-OUTLIERDETECTIONSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-outlierdetectionstep_0007.asdf
2026-05-18 07:28:06,503 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-18 07:28:06,579 - stpipe.step - INFO - Spec3Pipeline instance created.
2026-05-18 07:28:06,580 - stpipe.step - INFO - AssignMTWcsStep instance created.
2026-05-18 07:28:06,581 - stpipe.step - INFO - MasterBackgroundStep instance created.
2026-05-18 07:28:06,583 - stpipe.step - INFO - OutlierDetectionStep instance created.
2026-05-18 07:28:06,584 - stpipe.step - INFO - AdaptiveTraceModelStep instance created.
2026-05-18 07:28:06,584 - stpipe.step - INFO - PixelReplaceStep instance created.
2026-05-18 07:28:06,586 - stpipe.step - INFO - ResampleSpecStep instance created.
2026-05-18 07:28:06,588 - stpipe.step - INFO - C

Spec3 has been completed! 

Stage 3 for UGC0433+2251...
Applying Stage 3 Corrections & Calibrations to: clear_prism_l3asn.json


2026-05-18 07:28:35,452 - stpipe.step - INFO - PARS-OUTLIERDETECTIONSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-outlierdetectionstep_0007.asdf
2026-05-18 07:28:35,496 - stpipe.step - INFO - PARS-RESAMPLESPECSTEP parameters found: /home/dylan/.crds_cache/references/jwst/nirspec/jwst_nirspec_pars-resamplespecstep_0001.asdf
2026-05-18 07:28:35,575 - stpipe.step - INFO - Spec3Pipeline instance created.
2026-05-18 07:28:35,576 - stpipe.step - INFO - AssignMTWcsStep instance created.
2026-05-18 07:28:35,577 - stpipe.step - INFO - MasterBackgroundStep instance created.
2026-05-18 07:28:35,579 - stpipe.step - INFO - OutlierDetectionStep instance created.
2026-05-18 07:28:35,580 - stpipe.step - INFO - AdaptiveTraceModelStep instance created.
2026-05-18 07:28:35,581 - stpipe.step - INFO - PixelReplaceStep instance created.
2026-05-18 07:28:35,583 - stpipe.step - INFO - ResampleSpecStep instance created.
2026-05-18 07:28:35,584 - stpipe.step - INFO - C

Spec3 has been completed! 

